In [ ]:
learning_rate = 1e-4
batch_size = 128 # Minibatch size
epochs = 50
beta_momentum = 0.9 # Adam 1st moment decay (beta1)
beta_rmsprop = 0.999 # Adam 2nd moment decay (beta2)
epsilon = 1e-8 # small value to avoid division by zero
weight_decay = 1e-4 # AdamW decoupled weight decay

In [32]:
import numpy as np
BASE_PATH = '../../assets/raw'

def load_image(file):
    with open(file, 'rb') as f:
        f.seek(16)  # Skip the header
        data = np.frombuffer(f.read(), dtype=np.uint8)
    return data.reshape(-1, 28, 28)  # Reshape to (num_images, height, width)

def load_label(file):
    with open(file, 'rb') as f:
        f.seek(8)  # Skip the header
        data = np.frombuffer(f.read(), dtype=np.uint8)
    return data


In [33]:
def one_hot_encode(Y, num_classes):
    return np.eye(num_classes)[Y]

In [34]:
def split_mini_batches(X, Y, batch_size):
    m = X.shape[1]
    perm = np.random.permutation(m)
    X, Y_shuf = X[:, perm], Y[:, perm]
    return [(X[:, i:i + batch_size], Y_shuf[:, i:i + batch_size])
               for i in range(0, m, batch_size)]

In [35]:
def relu(Z):
    return np.maximum(0, Z)

In [36]:
def softmax(Z):
    exp_Z = np.exp(Z - np.max(Z, axis=0, keepdims=True)) # sub max for numerical stability
    return exp_Z / np.sum(exp_Z, axis=0, keepdims=True)

In [37]:
def relu_derivative(Z):
    return (Z > 0).astype(float) # derivative of ReLU (1 if Z > 0 else 0)

In [38]:
def forward_propagation(X, L, parameters):
    cache = {}
    A = X # A0 (input layer)

    for layer in range(1, L + 1):
        W = parameters[f'W{layer}']
        b = parameters[f'b{layer}']
        
        Z = np.dot(W, A) + b # Z = W * A + b
        if layer < L:
            A = relu(Z)
        else:
            A = softmax(Z) # output layer uses softmax for multi-class classification
        cache[f'Z{layer}'], cache[f'A{layer}'] = Z, A

    return A, cache

In [39]:
def loss(Y, Y_hat):
    m = Y.shape[1]
    return -np.sum(Y * np.log(np.clip(Y_hat, 1e-12, 1.0))) / m # Cross-entropy loss

In [40]:
def accuracy(Y, Y_hat):
    predictions = np.argmax(Y_hat, axis=0)
    labels = np.argmax(Y, axis=0)
    return np.mean(predictions == labels) * 100

In [41]:
def backward_propagation(X, dZ, m, L, parameters, cache):
    grads = {}

    for layer in reversed(range(1, L + 1)):
        A_prev = cache[f'A{layer - 1}'] if layer > 1 else X
        W = parameters[f'W{layer}']
        
        grads[f'dW{layer}'] = (1 / m) * np.dot(dZ, A_prev.T)
        grads[f'db{layer}'] = (1 / m) * np.sum(dZ, axis=1, keepdims=True)

        if layer > 1:
            dA_prev = np.dot(W.T, dZ)
            dZ = dA_prev * relu_derivative(cache[f'Z{layer - 1}'])

    return grads

### Initialization

In [42]:
X_train = load_image(f'{BASE_PATH}/train-images-idx3-ubyte').reshape(-1, 28*28).T  # Reshape to (num_features, num_samples)
Y_train = one_hot_encode(load_label(f'{BASE_PATH}/train-labels-idx1-ubyte'), num_classes=10).T # One-hot encode (1, num_samples) -> (num_classes, num_samples)

X_dev = load_image(f'{BASE_PATH}/t10k-images-idx3-ubyte').reshape(-1, 28*28).T  # Reshape to (num_features, num_samples)
Y_dev = one_hot_encode(load_label(f'{BASE_PATH}/t10k-labels-idx1-ubyte'), num_classes=10).T # One-hot encode (1, num_samples) -> (num_classes, num_samples)

In [43]:
X_train = X_train / 255.0  # Normalize pixel values to [0, 1]
X_dev = X_dev / 255.0 

In [44]:
units = [X_train.shape[0], 256, 128, Y_train.shape[0]] # input, hidden, output layers

parameters = {}
for layer in range(1, len(units)):
    # W: (layer units, layer-1 units)
    parameters[f'W{layer}'] = np.random.randn(units[layer], units[layer - 1]) * np.sqrt(2 / units[layer - 1]) # He initialization
    parameters[f'b{layer}'] = np.zeros((units[layer], 1))

adam_parameters = {}
for layer in range(1, len(units)):
    adam_parameters[f'VdW{layer}'] = np.zeros_like(parameters[f'W{layer}'])
    adam_parameters[f'Vdb{layer}'] = np.zeros_like(parameters[f'b{layer}'])
    adam_parameters[f'SdW{layer}'] = np.zeros_like(parameters[f'W{layer}'])
    adam_parameters[f'Sdb{layer}'] = np.zeros_like(parameters[f'b{layer}'])

### Model

In [ ]:
L = len(units) - 1 # number of layers (excluding input layer)
t = 0 # timestep for Adam optimizer

best_dev_cost = float('inf')
best_params = None
wait = 0

for epoch in range(epochs):
    batches = split_mini_batches(X_train, Y_train, batch_size)

    epoch_cost = 0
    for X_batch, Y_batch in batches:
        m = X_batch.shape[1]
        t += 1

        # forward prop
        A_L, cache = forward_propagation(X_batch, L, parameters)

        # loss
        cost = loss(Y_batch, A_L)
        epoch_cost += cost * m # accumulate weighted cost for the epoch

        # back prop
        dZ_L = A_L - Y_batch # derivative of loss w.r.t Z_L
        grads = backward_propagation(X_batch, dZ_L, m, L, parameters, cache)

        # update parameters
        for layer in range(1, L + 1):
            # adam update
            adam_parameters[f'VdW{layer}'] = beta_momentum * adam_parameters[f'VdW{layer}'] + (1 - beta_momentum) * grads[f'dW{layer}']
            adam_parameters[f'Vdb{layer}'] = beta_momentum * adam_parameters[f'Vdb{layer}'] + (1 - beta_momentum) * grads[f'db{layer}']

            adam_parameters[f'SdW{layer}'] = beta_rmsprop * adam_parameters[f'SdW{layer}'] + (1 - beta_rmsprop) * (grads[f'dW{layer}'] ** 2)
            adam_parameters[f'Sdb{layer}'] = beta_rmsprop * adam_parameters[f'Sdb{layer}'] + (1 - beta_rmsprop) * (grads[f'db{layer}'] ** 2)

            VdW_corrected = adam_parameters[f'VdW{layer}'] / (1 - beta_momentum ** t)
            Vdb_corrected = adam_parameters[f'Vdb{layer}'] / (1 - beta_momentum ** t)

            SdW_corrected = adam_parameters[f'SdW{layer}'] / (1 - beta_rmsprop ** t)
            Sdb_corrected = adam_parameters[f'Sdb{layer}'] / (1 - beta_rmsprop ** t)

            parameters[f'W{layer}'] -= learning_rate * (
                VdW_corrected / (np.sqrt(SdW_corrected) + epsilon) + weight_decay * parameters[f'W{layer}']
            )
            parameters[f'b{layer}'] -= learning_rate * (
                Vdb_corrected / (np.sqrt(Sdb_corrected) + epsilon)
            )
    epoch_cost /= X_train.shape[1]  # average cost for the epoch

    if epoch % 10 == 0 or epoch == epochs - 1:
        # evaluate on dev set
        A_dev, _ = forward_propagation(X_dev, L, parameters)
        A_train, _ = forward_propagation(X_train, L, parameters)

        dev_cost = loss(Y_dev, A_dev)
        
        dev_acc = accuracy(Y_dev, A_dev)
        train_acc = accuracy(Y_train, A_train)
        print(f'Epoch {epoch + 1}/{epochs}, Training Loss: {epoch_cost:.4f}, Dev Loss: {dev_cost:.4f}, Train Acc: {train_acc:.2f}%, Dev Acc: {dev_acc:.2f}%')
    
    if dev_cost < best_dev_cost:
        best_dev_cost = dev_cost
        best_params = {k: v.copy() for k, v in parameters.items()}  # Save best parameters
        wait = 0  # Reset wait counter
    else:
        wait += 1
        if wait >= 20:  # Early stopping patience
            print(f"Early stopping at epoch {epoch + 1}. Best Dev Loss: {best_dev_cost:.4f}")
            break

Epoch 1/250, Training Loss: 0.5878, Dev Loss: 0.4508, Train Acc: 85.65%, Dev Acc: 84.29%
Epoch 11/250, Training Loss: 0.2417, Dev Loss: 0.3214, Train Acc: 92.19%, Dev Acc: 88.38%
Epoch 21/250, Training Loss: 0.1759, Dev Loss: 0.3231, Train Acc: 94.13%, Dev Acc: 89.04%
Early stopping at epoch 21. Best Dev Loss: 0.3214
